# Facsimile + Measure-Zone Pipeline Workflow (Workflow 1)

Maintainer notebook for the packaged `camat-run-pipeline` workflow: preflight counts, build the exact command, optionally run selected network/write steps, and summarize the newest reports. Corpus data and reports remain in the external edition repository. All network and write actions are disabled by default.

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

ROOT = Path.cwd()
print(ROOT)

## Configuration

In [ ]:
SCORE_DIR = Path("DdT_1/29_30_instrumentalkonzerte_deutscher_meister_bsb00023250")

# Full normal run: STEPS = None, OVERWRITE_IMAGES = False
# Recovery from stale/mismatched local JPEGs: STEPS = [1, 3, 4, 5], OVERWRITE_IMAGES = True
STEPS: list[int] | None = None
OVERWRITE_IMAGES = False
MINIMUM_MEASURES = 5
MAX_MEASURE_MISMATCH = 1
TIMEOUT = 180
FORCE = False  # True allows any amount of source-vs-annotation measure mismatch
PAGES = None  # e.g. "30-" to run only pages 30 onward
SKIP_PAGES = "1-29"  # preface pages that should not be processed/validated

# Deliberately off by default. Set True when you want this notebook to launch the long network run.
RUN_PIPELINE = False

if not SCORE_DIR.exists():
    print(f"Configure SCORE_DIR; it does not exist yet: {SCORE_DIR}")
SCORE_DIR

## Preflight

In [ ]:
def source_mei_files(score_dir: Path) -> list[Path]:
    return sorted(p for p in score_dir.glob("*.mei") if not p.stem.endswith("_facs_zones"))


def inventory(score_dir: Path) -> dict[str, object]:
    src = source_mei_files(score_dir)
    missing_outputs = [p.stem for p in src if not (score_dir / f"{p.stem}_facs_zones.mei").exists()]
    return {
        "source_mei": len(src),
        "images": len(list((score_dir / "img").glob("*.jpg"))) if (score_dir / "img").exists() else 0,
        "annotations": len(list(score_dir.glob("*_measure_annotations.xml"))),
        "facs_zones": len(list(score_dir.glob("*_facs_zones.mei"))),
        "missing_outputs": len(missing_outputs),
        "missing_output_head": missing_outputs[:20],
    }


inventory(SCORE_DIR)

## BSB Page Coverage

In [ ]:
RUN_COVERAGE_CHECK = False
COVERAGE_WRITE_REPORTS = False

coverage_cmd = [sys.executable, "-m", "camat.check_bsb_page_coverage", str(SCORE_DIR)]
if not COVERAGE_WRITE_REPORTS:
    coverage_cmd.append("--no-write")

print(" ".join(coverage_cmd))
if RUN_COVERAGE_CHECK:
    subprocess.run(coverage_cmd, cwd=ROOT, check=False)
else:
    print("RUN_COVERAGE_CHECK is False. Set it to True to query the BSB manifest.")

## Build And Run Command

In [ ]:
def pipeline_command(
    score_dir: Path,
    *,
    steps: list[int] | None = None,
    overwrite_images: bool = False,
    minimum_measures: int = 5,
    max_measure_mismatch: int = 1,
    timeout: int = 180,
    force: bool = False,
    pages: str | None = None,
    skip_pages: str | None = None,
) -> list[str]:
    cmd = [sys.executable, "-m", "camat.run_pipeline", str(score_dir)]
    if overwrite_images:
        cmd.append("--overwrite-images")
    if steps:
        cmd.extend(["--steps", *map(str, steps)])
    cmd.extend([
        "--minimum-measures",
        str(minimum_measures),
        "--max-measure-mismatch",
        str(max_measure_mismatch),
        "--timeout",
        str(timeout),
    ])
    if force:
        cmd.append("--force")
    if pages:
        cmd.extend(["--pages", pages])
    if skip_pages:
        cmd.extend(["--skip-pages", skip_pages])
    return cmd


cmd = pipeline_command(
    SCORE_DIR,
    steps=STEPS,
    overwrite_images=OVERWRITE_IMAGES,
    minimum_measures=MINIMUM_MEASURES,
    max_measure_mismatch=MAX_MEASURE_MISMATCH,
    timeout=TIMEOUT,
    force=FORCE,
    pages=PAGES,
    skip_pages=SKIP_PAGES,
)
print(" ".join(cmd))

In [ ]:
if RUN_PIPELINE:
    subprocess.run(cmd, cwd=ROOT, check=True)
else:
    print("RUN_PIPELINE is False. Set it to True above to execute the command.")

## Direct Integration Command

Optional lower-level command for rerunning only the packaged annotation upload/integration module without the other pipeline steps.

In [ ]:
CUSTOM_MINIMUM_MEASURES = MINIMUM_MEASURES
CUSTOM_MAX_MEASURE_MISMATCH = MAX_MEASURE_MISMATCH
CUSTOM_GRAPHIC_TARGET_MODE = "iiif"  # "local" or "iiif"
CUSTOM_FORCE = FORCE
CUSTOM_PAGES = PAGES
CUSTOM_SKIP_PAGES = SKIP_PAGES
RUN_CUSTOM_INTEGRATION = False
RUN_CUSTOM_VALIDATION = False

In [ ]:
def custom_integration_command(
    score_dir: Path,
    *,
    minimum_measures: int,
    max_measure_mismatch: int,
    graphic_target_mode: str,
    force: bool = False,
    pages: str | None = None,
    skip_pages: str | None = None,
) -> list[str]:
    cmd = [
        sys.executable,
        "-m",
        "camat.upload_and_integrate_measure_annotations",
        str(score_dir),
        "--minimum-measures",
        str(minimum_measures),
        "--max-measure-mismatch",
        str(max_measure_mismatch),
        "--timeout",
        "180",
        "--reuse-annotations",
        "--overwrite",
        "--graphic-target-mode",
        graphic_target_mode,
    ]
    if force:
        cmd.append("--force")
    if pages:
        cmd.extend(["--pages", pages])
    if skip_pages:
        cmd.extend(["--skip-pages", skip_pages])
    return cmd


custom_cmd = custom_integration_command(
    SCORE_DIR,
    minimum_measures=CUSTOM_MINIMUM_MEASURES,
    max_measure_mismatch=CUSTOM_MAX_MEASURE_MISMATCH,
    graphic_target_mode=CUSTOM_GRAPHIC_TARGET_MODE,
    force=CUSTOM_FORCE,
    pages=CUSTOM_PAGES,
    skip_pages=CUSTOM_SKIP_PAGES,
)
print(" ".join(custom_cmd))

In [ ]:
if RUN_CUSTOM_INTEGRATION:
    subprocess.run(custom_cmd, cwd=ROOT, check=True)
else:
    print("RUN_CUSTOM_INTEGRATION is False. Set it to True above to execute the custom integration.")

In [ ]:
custom_validation_cmd = [sys.executable, "-m", "camat.validate_iiif_vs_local", str(SCORE_DIR), "--check-output-mei"]
if CUSTOM_PAGES:
    custom_validation_cmd.extend(["--pages", CUSTOM_PAGES])
if CUSTOM_SKIP_PAGES:
    custom_validation_cmd.extend(["--skip-pages", CUSTOM_SKIP_PAGES])
print(" ".join(custom_validation_cmd))

if RUN_CUSTOM_VALIDATION:
    subprocess.run(custom_validation_cmd, cwd=ROOT, check=True)
else:
    print("RUN_CUSTOM_VALIDATION is False. Set it to True after integration to validate outputs.")

## Latest Reports

In [ ]:
def latest_json(score_dir: Path, pattern: str) -> Path | None:
    reports = sorted(score_dir.glob(pattern))
    return reports[-1] if reports else None


def load_json(path: Path | None) -> dict:
    return json.loads(path.read_text(encoding="utf-8")) if path else {}


latest_measure_report = latest_json(SCORE_DIR, "measure_annotation_run_*.json")
latest_validation_report = latest_json(SCORE_DIR, "iiif_validation_*.json")

print("latest measure report:", latest_measure_report)
print("latest validation report:", latest_validation_report)

In [ ]:
measure_data = load_json(latest_measure_report)
settings = measure_data.get("settings", {})

summary = {
    "counts": measure_data.get("counts"),
    "measure_filters": {
        "minimum_measures": settings.get("minimum_measures"),
        "max_measure_mismatch": settings.get("max_measure_mismatch"),
    },
    "graphic_target_mode": settings.get("graphic_target_mode"),
    "reuse_annotations": settings.get("reuse_annotations"),
    "overwrite": settings.get("overwrite"),
    "aborted": measure_data.get("aborted"),
}

summary

In [ ]:
failed = measure_data.get("failed_files", [])
failure_categories = Counter(
    "HTTP 500" if "HTTP 500" in item.get("reason", "")
    else "measure_mismatch" if "Measure numbering" in item.get("reason", "")
    else "other"
    for item in failed
)

print("skipped:", len(measure_data.get("skipped_files", [])))
print("failed:", len(failed), failure_categories)
print("HTTP 500 files:", [item["file"] for item in failed if "HTTP 500" in item.get("reason", "")])
print("first mismatches:")
for item in [x for x in failed if "Measure numbering" in x.get("reason", "")][:10]:
    print("-", item["file"], item["reason"][:180])

In [ ]:
validation_data = load_json(latest_validation_report)
validation_summary = validation_data.get("summary", {})
validation_failed = validation_data.get("failed", [])

print(validation_summary)
print("failed entries:", len(validation_failed))
print("first failed:", validation_failed[:5])

## Manual Review List

In [ ]:
import csv

EXPORT_MANUAL_REVIEW = False


def manual_review_rows(score_dir: Path, measure_report: dict, validation_report: dict) -> list[dict[str, str]]:
    rows: list[dict[str, str]] = []

    def add_row(file_name: str, status: str, reason: str) -> None:
        stem = Path(file_name).stem
        rows.append({
            "status": status,
            "file": file_name,
            "stem": stem,
            "reason": reason,
            "mei_path": str(score_dir / f"{stem}.mei"),
            "image_path": str(score_dir / "img" / f"{stem}.jpg"),
            "annotation_path": str(score_dir / f"{stem}_measure_annotations.xml"),
            "output_path": str(score_dir / f"{stem}_facs_zones.mei"),
        })

    for item in measure_report.get("failed_files", []):
        add_row(item.get("file", ""), "integration_failed", item.get("reason", ""))
    for item in measure_report.get("skipped_files", []):
        add_row(item.get("file", ""), "integration_skipped", item.get("reason", ""))

    seen_files = {row["file"] for row in rows}
    for item in validation_report.get("failed", []):
        file_name = item.get("file", "")
        if file_name not in seen_files:
            add_row(file_name, "validation_failed", item.get("error", ""))

    return rows


review_rows = manual_review_rows(SCORE_DIR, measure_data, validation_data)
print("manual review rows:", len(review_rows))
for row in review_rows[:20]:
    print(row["status"], row["file"], "-", row["reason"][:140])


In [ ]:
report_stem = latest_measure_report.stem if latest_measure_report else "measure_annotation_run"
review_csv = SCORE_DIR / f"{report_stem}_manual_review.csv"
review_txt = SCORE_DIR / f"{report_stem}_failed_meis.txt"

if not review_rows:
    print("No failed/skipped rows to export.")
elif not EXPORT_MANUAL_REVIEW:
    print("EXPORT_MANUAL_REVIEW is False. Set it to True to write the review files.")
else:
    with review_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(review_rows[0].keys()))
        writer.writeheader()
        writer.writerows(review_rows)

    review_txt.write_text("\n".join(row["mei_path"] for row in review_rows) + "\n", encoding="utf-8")

    print(review_csv)
    print(review_txt)


## Spot-Check IIIF Output

In [ ]:
def first_output_with_iiif(score_dir: Path) -> tuple[Path | None, str | None]:
    for path in sorted(score_dir.glob("*_facs_zones.mei")):
        text = path.read_text(encoding="utf-8", errors="replace")
        marker = "https://api.digitale-sammlungen.de/iiif/image/v2/"
        if marker in text:
            line = next((line.strip() for line in text.splitlines() if marker in line), None)
            return path, line
    return None, None


path, line = first_output_with_iiif(SCORE_DIR)
print(path)
print(line)